# Context-Aware Music Recommender System

A six-stage pipeline combining content-based filtering, collaborative filtering, and a hybrid model with SHAP/LIME explainability.

---
## Stage 1 — Data Collection

Two datasets are used:
- **Last.fm 1K** — timestamped listening histories (artist ID, track name, user ID)
- **Spotify dataset** — audio features per track (tempo, energy, valence, danceability, acousticness, etc.)

Last.fm provides behavioural interaction data; Spotify provides the audio characteristics needed for content-based filtering.

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
import shap
import lime
import lime.lime_tabular
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix
from collections import defaultdict
warnings.filterwarnings('ignore')

/home/mont/Desktop/code/music-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Last.fm 1K dataset — 500k rows loaded (full dataset is 19M rows)
columns = ['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname']

df_lastfm = pd.read_csv(
    'data/lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv',
    sep='\t',
    on_bad_lines='skip',
    names=columns,
    header=None,
    nrows=500000
)

print(f"Last.fm shape: {df_lastfm.shape}")
df_lastfm.head(3)

Last.fm shape: (500000, 6)


,userid,timestamp,artid,artname,traid,traname
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15)
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15)


In [3]:
# Spotify audio features dataset (Kaggle)
df_spotify = pd.read_csv('data/spotify-dataset/data.csv')

print(f"Spotify shape: {df_spotify.shape}")
df_spotify.head(3)

Spotify shape: (170653, 19)


,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878,10,0.665,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.732,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000,7,0.160,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936
2,0.0394,1921,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913,3,0.101,-14.850,1,Gati Bali,5,1921,0.0339,110.339


---
## Stage 2 — Data Analysis & Cleaning

Key questions examined before preprocessing:
- What is the match rate between Last.fm tracks and Spotify entries?
- How sparse is the user-item play count matrix?
- How skewed is the play count distribution?
- What proportion of tracks require audio feature imputation?

In [4]:
# Normalise track and artist names for fuzzy-tolerant exact matching
def normalise(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

df_spotify['name_norm']    = df_spotify['name'].apply(normalise)
df_spotify['artists_norm'] = df_spotify['artists'].apply(normalise)
df_lastfm['traname_norm']  = df_lastfm['traname'].apply(normalise)
df_lastfm['artname_norm']  = df_lastfm['artname'].apply(normalise)

merged = df_lastfm.merge(
    df_spotify,
    left_on=['artname_norm', 'traname_norm'],
    right_on=['artists_norm', 'name_norm'],
    how='left'
)

match_rate = merged['tempo'].notna().mean()
print(f"Spotify match rate: {match_rate:.1%}")
print(f"Tracks requiring audio feature imputation: {1 - match_rate:.1%}")

Spotify match rate: 70.6%
Tracks requiring audio feature imputation: 29.4%


In [5]:
# User-item matrix sparsity
n_users  = df_lastfm['userid'].nunique()
n_tracks = df_lastfm['traname'].nunique()
n_interactions = df_lastfm.groupby(['userid', 'traname']).ngroups
sparsity = 1 - n_interactions / (n_users * n_tracks)

print(f"Users: {n_users}")
print(f"Unique tracks: {n_tracks:,}")
print(f"Observed user-track pairs: {n_interactions:,}")
print(f"Matrix sparsity: {sparsity:.4%}")

Users: 22
Unique tracks: 76,620
Observed user-track pairs: 99,880
Matrix sparsity: 94.0747%


In [6]:
# Play count distribution — check for heavy right skew
user_plays = df_lastfm.groupby('userid').size()
print("Play count distribution per user:")
print(user_plays.describe())

Play count distribution per user:
count       22.000000
mean     22727.272727
std      20645.541897
min        354.000000
25%       9600.500000
50%      17791.000000
75%      26187.500000
max      75876.000000
dtype: float64


---
## Stage 3 — Preprocessing & Feature Engineering

Steps applied:
1. Parse timestamps → extract `time_of_day`, `day_of_week`, `day_type`
2. Group consecutive events into sessions (30-minute inactivity threshold)
3. Impute missing audio features using artist-level medians, then global median fallback
4. Scale all audio features to [0, 1] with MinMaxScaler
5. Derive per-track time-of-day listening distributions

In [7]:
# Temporal feature extraction
def get_time_of_day(hour):
    if 5 <= hour < 12:   return 'morning'
    elif 12 <= hour < 17: return 'afternoon'
    elif 17 <= hour < 21: return 'evening'
    else:                 return 'night'

df_lastfm['datetime']    = pd.to_datetime(df_lastfm['timestamp'], utc=True)
df_lastfm['hour']        = df_lastfm['datetime'].dt.hour
df_lastfm['time_of_day'] = df_lastfm['hour'].apply(get_time_of_day)
df_lastfm['day_of_week'] = df_lastfm['datetime'].dt.weekday
df_lastfm['day_type']    = df_lastfm['day_of_week'].apply(lambda x: 'weekend' if x >= 5 else 'weekday')

print("Time-of-day distribution:")
print(df_lastfm['time_of_day'].value_counts())

Time-of-day distribution:
time_of_day
night        174541
afternoon    125053
evening      114424
morning       85982
Name: count, dtype: int64


In [8]:
merged['time_of_day'] = merged['userid'].map(
    df_lastfm.set_index(df_lastfm.index)['time_of_day']
)

In [9]:
# Session segmentation — gap > 30 minutes triggers a new session
SESSION_GAP = 30 * 60  # seconds

df_lastfm = df_lastfm.sort_values(['userid', 'datetime'])
df_lastfm['time_diff']   = df_lastfm.groupby('userid')['datetime'].diff().dt.seconds
df_lastfm['new_session'] = (df_lastfm['time_diff'] > SESSION_GAP) | (df_lastfm['time_diff'].isna())
df_lastfm['session_id']  = df_lastfm.groupby('userid')['new_session'].cumsum()

total_sessions = df_lastfm.groupby('userid')['session_id'].max().sum()
print(f"Total sessions identified: {total_sessions:,}")

Total sessions identified: 26,446


In [10]:
# Build per-track feature matrix: audio features + time-of-day distribution
AUDIO_FEATURES = [
    'acousticness', 'danceability', 'energy',
    'instrumentalness', 'loudness', 'speechiness',
    'tempo', 'valence', 'popularity'
]
TIME_SLOTS = ['morning', 'afternoon', 'evening', 'night']

# Aggregate to one row per unique (artist, track)
tracks = merged.groupby(['artname', 'traname']).agg(
    total_plays=('userid', 'count'),
    **{f: (f, 'mean') for f in AUDIO_FEATURES}
).reset_index()

# Impute missing audio features: artist median → global median
for feat in AUDIO_FEATURES:
    artist_means = tracks.groupby('artname')[feat].transform('mean')
    tracks[feat] = tracks[feat].fillna(artist_means)
    tracks[feat] = tracks[feat].fillna(tracks[feat].median())

# Scale audio features to [0, 1]
scaler = MinMaxScaler()
tracks[AUDIO_FEATURES] = scaler.fit_transform(tracks[AUDIO_FEATURES])

# Time-of-day distribution per track (normalised listen counts)
tod_counts = (
    merged.groupby(['artname', 'traname', 'time_of_day'])
    .size().unstack(fill_value=0)
    .reindex(columns=TIME_SLOTS, fill_value=0)
)
tod_counts = tod_counts.div(tod_counts.sum(axis=1).replace(0, 1), axis=0)
tracks = tracks.merge(tod_counts.reset_index(), on=['artname', 'traname'], how='left')
tracks[TIME_SLOTS] = tracks[TIME_SLOTS].fillna(0.25)
tracks['track_id'] = range(len(tracks))

# Sparse item matrix: audio (9) + time-of-day (4) = 13 features
audio_matrix = csr_matrix(tracks[AUDIO_FEATURES].values)
tod_matrix   = csr_matrix(tracks[TIME_SLOTS].values)
item_matrix  = hstack([audio_matrix, tod_matrix])

print(f"Track catalogue size: {len(tracks):,}")
print(f"Item matrix shape: {item_matrix.shape}")

Track catalogue size: 88,695
Item matrix shape: (88695, 13)


---
## Stage 4 — Recommendation Engine

Three models are implemented independently against the same preprocessed data.

### 4.1 Content-Based Filtering

Builds a weighted user profile vector from play-count-weighted track features, then ranks all candidate tracks by cosine similarity. A 25% context boost is applied when a track's dominant time-of-day slot matches the current session.

In [11]:
BOOST = 1.25  # 25% context lift

FEATURE_LABELS = {
    'energy':           'high energy',
    'acousticness':     'acoustic feel',
    'danceability':     'danceability',
    'valence':          'positive mood',
    'instrumentalness': 'instrumental style',
    'tempo':            'tempo',
}

def build_user_profile(userid, df, tracks, item_matrix):
    """Weighted average of all tracks the user has played."""
    user_plays = (
        df[df['userid'] == userid]
        .groupby(['artname', 'traname'])
        .size().reset_index(name='play_count')
    )
    user_plays = user_plays.merge(
        tracks[['artname', 'traname', 'track_id']],
        on=['artname', 'traname'], how='inner'
    )
    if user_plays.empty:
        return None, None
    weights = user_plays['play_count'].values.astype(float)
    weights /= weights.sum()
    profile = csr_matrix(weights) @ item_matrix[user_plays['track_id'].values]
    return profile, set(zip(user_plays['artname'], user_plays['traname']))


def get_candidates(userid, df, tracks):
    """Return tracks the user has not yet heard."""
    heard = set(zip(
        df[df['userid'] == userid]['artname'],
        df[df['userid'] == userid]['traname']
    ))
    mask = ~(pd.Series(list(zip(tracks['artname'], tracks['traname']))).isin(heard))
    return tracks[mask]


def score_candidates(user_profile, candidates, item_matrix):
    """Cosine similarity between user profile and each candidate track."""
    cand_vectors = item_matrix[candidates['track_id'].values]
    scores = cosine_similarity(user_profile, cand_vectors).flatten()
    candidates = candidates.copy()
    candidates['base_score'] = scores
    return candidates


def apply_context_boost(candidates, current_time_of_day):
    """Apply 25% score lift when track's dominant TOD matches current session."""
    candidates = candidates.copy()
    candidates['dominant_tod'] = candidates[TIME_SLOTS].idxmax(axis=1)
    match = candidates['dominant_tod'] == current_time_of_day
    candidates['final_score'] = candidates['base_score'] * np.where(match, BOOST, 1.0)
    return candidates


def recommend(userid, current_time_of_day, df, tracks, item_matrix, n=10):
    """Content-based recommendation — excludes already-heard tracks."""
    profile, _ = build_user_profile(userid, df, tracks, item_matrix)
    if profile is None:
        return pd.DataFrame()
    candidates = get_candidates(userid, df, tracks)
    candidates = score_candidates(profile, candidates, item_matrix)
    candidates = apply_context_boost(candidates, current_time_of_day)
    return candidates.sort_values('final_score', ascending=False).head(n)

In [12]:
# Demo: content-based recommendations for user_000001
USER = 'user_000001'
CURRENT_TIME_OF_DAY = 'night'

cb_demo = recommend(USER, CURRENT_TIME_OF_DAY, df_lastfm, tracks, item_matrix)
print(cb_demo[['artname', 'traname', 'final_score']].to_string())

      artname                                       traname  final_score
57358  R.E.M.                              Man-Sized Wreath     0.999018
57354  R.E.M.                            Love Is All Around     0.999018
57357  R.E.M.               Man On The Moon (Album Version)     0.999018
57377  R.E.M.                                 The Apologist     0.999018
57381  R.E.M.                         Until The Day Is Done     0.999018
57367  R.E.M.  Radio Free Europe (Original Hib-Tone Single)     0.999018
57319  R.E.M.                     Airportman ( Lp Version )     0.999018
57345  R.E.M.             Imitation Of Life (Album Version)     0.999018
57370  R.E.M.                        Sing For The Submarine     0.999018
57371  R.E.M.                                         Stand     0.999018


### 4.2 Collaborative Filtering

Builds a user-item play count matrix and decomposes it via TruncatedSVD (50 latent components) to recover user and item factors. Predicted scores are reconstructed from the dot product of these factors.

In [13]:
# Build user-item play count matrix
user_item = (
    df_lastfm.groupby(['userid', 'traname'])
    .size()
    .reset_index(name='play_count')
)
ui_pivot  = user_item.pivot(index='userid', columns='traname', values='play_count').fillna(0)
ui_sparse = csr_matrix(ui_pivot.values)

user_index  = {u: i for i, u in enumerate(ui_pivot.index)}
track_index = {t: i for i, t in enumerate(ui_pivot.columns)}

# Latent factor decomposition — 50 components
svd          = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(ui_sparse)   # (n_users, 50)
item_factors = svd.components_.T              # (n_tracks, 50)
predicted    = user_factors @ item_factors.T  # (n_users, n_tracks)

print(f"User-item matrix: {ui_sparse.shape}")
print(f"SVD explained variance: {svd.explained_variance_ratio_.sum():.2%}")

User-item matrix: (22, 76620)
SVD explained variance: 100.00%


In [14]:
def recommend_cf(userid, df, n=10):
    """Collaborative filtering recommendation — excludes already-heard tracks."""
    if userid not in user_index:
        return pd.DataFrame()  # cold-start: no CF signal
    u_idx = user_index[userid]
    scores = predicted[u_idx]
    heard_tracks = set(df[df['userid'] == userid]['traname'])
    results = [
        {'traname': track, 'cf_score': scores[t_idx]}
        for track, t_idx in track_index.items()
        if track not in heard_tracks
    ]
    return pd.DataFrame(results).sort_values('cf_score', ascending=False).head(n)


# Demo: CF recommendations for user_000001
cf_demo = recommend_cf(USER, df_lastfm)
print(cf_demo.to_string())

                     traname      cf_score
3252                   Alive  2.512971e-13
49345         Revolution 909  2.301525e-13
13820                Da Funk  2.198775e-13
7635   Bitter Sweet Symphony  2.160975e-13
9970              By The Way  2.109253e-13
21305          Forget Myself  2.096089e-13
55074        Soul Meets Body  1.999517e-13
67593    Walking On The Sand  1.990084e-13
19827         Fancy Footwork  1.986430e-13
57217            Summer Skin  1.975840e-13


### 4.3 Hybrid Model

Combines content-based and collaborative scores using fixed weights (α=0.3 CB, β=0.7 CF). Both scores are rank-normalised to percentiles before blending to ensure comparable scales. An `exclude_heard` flag switches between production mode (filter heard tracks) and evaluation mode (score all tracks, since ground truth includes re-listens).

In [15]:
ALPHA = 0.3  # content-based weight
BETA  = 0.7  # collaborative filtering weight

def recommend_hybrid(userid, current_time_of_day, df, tracks, item_matrix, n=10,
                     _user_index=None, _predicted=None, _track_index=None,
                     exclude_heard=True):
    """
    Hybrid recommender blending CB and CF scores.
    Pass _user_index/_predicted/_track_index to use a train-split CF model during eval.
    Set exclude_heard=False during evaluation (ground truth includes re-listens).
    """
    _user_index  = _user_index  if _user_index  is not None else user_index
    _predicted   = _predicted   if _predicted   is not None else predicted
    _track_index = _track_index if _track_index is not None else track_index

    # --- Content-based scores ---
    if exclude_heard:
        cb_results = recommend(userid, current_time_of_day, df, tracks, item_matrix, n=2000)
    else:
        # Eval mode: score only tracks known to the CF model
        profile, _ = build_user_profile(userid, df, tracks, item_matrix)
        if profile is None:
            return pd.DataFrame()
        cf_known_tracks = tracks[tracks['traname'].isin(_track_index.keys())]
        cb_results = score_candidates(profile, cf_known_tracks, item_matrix)
        cb_results = apply_context_boost(cb_results, current_time_of_day)
        cb_results = cb_results.sort_values('final_score', ascending=False)

    if cb_results.empty:
        return pd.DataFrame()

    # Rank-normalise CB scores to percentile [0, 1]
    cb_results['cb_norm'] = cb_results['final_score'].rank(pct=True)

    # --- CF scores ---
    if userid in _user_index:
        u_idx = _user_index[userid]
        cf_row = _predicted[u_idx]
        cb_results['cf_norm'] = cb_results['traname'].map(
            lambda t: cf_row[_track_index[t]] if t in _track_index else 0.0
        )
        # Rank-normalise CF scores to percentile [0, 1]
        cb_results['cf_norm'] = cb_results['cf_norm'].rank(pct=True)
    else:
        cb_results['cf_norm'] = 0.0  # cold-start fallback

    cb_results['hybrid_score'] = ALPHA * cb_results['cb_norm'] + BETA * cb_results['cf_norm']
    return cb_results.sort_values('hybrid_score', ascending=False).head(n)

In [16]:
# Demo: hybrid recommendations for user_000001
hybrid_results = recommend_hybrid(USER, CURRENT_TIME_OF_DAY, df_lastfm, tracks, item_matrix, exclude_heard=False)
print(hybrid_results[['artname', 'traname', 'hybrid_score']].to_string())

            artname                 traname  hybrid_score
9853           Blur               Movin' On      0.992899
63576         Slade        How Does It Feel      0.984464
35347  Jimi Hendrix                Midnight      0.978449
26911   Frank Zappa          Plastic People      0.976433
9911           Blur              Turn It Up      0.973263
9771           Blur               Best Days      0.963603
57361        R.E.M.                  Munich      0.958312
79949       The Who               Magic Bus      0.955956
9760           Blur                 Bad Day      0.954022
79998       The Who  Won'T Get Fooled Again      0.953446


---
## Stage 5 — Explanation Layer

Two post-hoc explainability methods are applied to the hybrid model:
- **SHAP** (KernelExplainer) — quantifies each feature's contribution to the recommendation score
- **LIME** (LimeTabularExplainer) — independently fits a local linear surrogate per track

Features that both methods agree on are presented as **confident drivers**. Disagreements flag recommendations where the explanation may be unstable.

In [17]:
# Shared setup: scorer function and background reference set
from sklearn.metrics.pairwise import cosine_similarity as _cos

ALL_FEATURES = AUDIO_FEATURES + TIME_SLOTS  # 13 features matching item_matrix column order

def _profile_dense(userid):
    """Dense user profile vector (shape 13)."""
    profile_sparse, _ = build_user_profile(userid, df_lastfm, tracks, item_matrix)
    if profile_sparse is None:
        raise ValueError(f'No history for {userid}')
    return profile_sparse.toarray().flatten()

def _make_scorer(profile_1d):
    """Returns a scorer f(X [n,13]) -> scores [n] for SHAP/LIME."""
    p = profile_1d.reshape(1, -1)
    return lambda X: _cos(p, X).flatten()

def _background(userid, n=100):
    """Sample of heard tracks as dense numpy array — used as SHAP/LIME baseline."""
    heard = set(zip(df_lastfm[df_lastfm['userid']==userid]['artname'],
                    df_lastfm[df_lastfm['userid']==userid]['traname']))
    ht  = tracks[tracks.apply(lambda r: (r['artname'], r['traname']) in heard, axis=1)]
    mat = ht[ALL_FEATURES].values.astype(float)
    if len(mat) > n:
        mat = mat[np.random.choice(len(mat), n, replace=False)]
    return mat

profile_dense = _profile_dense(USER)
scorer        = _make_scorer(profile_dense)
background    = _background(USER, n=100)
print(f"Profile shape: {profile_dense.shape}  |  Background: {background.shape}")

Profile shape: (13,)  |  Background: (100, 13)


### 5.1 SHAP — Feature Contribution Scores

`KernelExplainer` perturbs each track's feature vector and measures how much each feature pushes the cosine-similarity score **up (+) or down (−)** relative to the user's heard-tracks baseline.

In [18]:
print('Building SHAP explainer …')
shap_explainer = shap.KernelExplainer(scorer, background)
print('Done.')

shap_rows = []
for _, row in hybrid_results.iterrows():
    x  = row[ALL_FEATURES].values.reshape(1, -1).astype(float)
    sv = shap_explainer.shap_values(x, silent=True)[0]
    entry = dict(zip(ALL_FEATURES, sv))
    entry.update({'artname': row['artname'], 'traname': row['traname'],
                  'hybrid_score': row['hybrid_score']})
    shap_rows.append(entry)

shap_df = pd.DataFrame(shap_rows)

# Waterfall chart for top recommendation
top_row  = hybrid_results.iloc[0]
top_shap = shap_df.iloc[0]
sv_vals  = np.array([top_shap[f] for f in ALL_FEATURES])
order    = np.argsort(np.abs(sv_vals))[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
colors  = ['#2ecc71' if v > 0 else '#e74c3c' for v in sv_vals[order]]
ax.barh([ALL_FEATURES[i] for i in order][::-1], sv_vals[order][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value  (impact on cosine-similarity score)')
ax.set_title(f"SHAP — {top_row['artname']} · {top_row['traname']}")
plt.tight_layout()
plt.show()

Building SHAP explainer …
Done.


### 5.2 LIME — Cross-Validation of SHAP

`LimeTabularExplainer` fits a local linear model around each recommendation. Its coefficients independently confirm or dispute SHAP's top features.

In [19]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = background,
    feature_names         = ALL_FEATURES,
    mode                  = 'regression',
    discretize_continuous = False,
    random_state          = 42
)

def _lime_coefs(track_row):
    x   = track_row[ALL_FEATURES].values.astype(float)
    exp = lime_explainer.explain_instance(
        data_row=x, predict_fn=scorer,
        num_features=len(ALL_FEATURES), num_samples=500
    )
    coefs = {}
    for label, coef in exp.as_list():
        for feat in ALL_FEATURES:
            if feat in label:
                coefs[feat] = coef
                break
    return coefs

print('Running LIME explanations …')
lime_rows = []
for _, row in hybrid_results.iterrows():
    lv = _lime_coefs(row)
    lv.update({'artname': row['artname'], 'traname': row['traname']})
    lime_rows.append(lv)
lime_df = pd.DataFrame(lime_rows)
print('Done.')

# Bar chart for top recommendation
top_lime = lime_df.iloc[0]
lv_vals  = np.array([top_lime.get(f, 0) for f in ALL_FEATURES])
l_order  = np.argsort(np.abs(lv_vals))[::-1]

fig2, ax2 = plt.subplots(figsize=(9, 5))
lcolors = ['#3498db' if v > 0 else '#e67e22' for v in lv_vals[l_order]]
ax2.barh([ALL_FEATURES[i] for i in l_order][::-1], lv_vals[l_order][::-1], color=lcolors[::-1])
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('LIME coefficient  (local linear approximation)')
ax2.set_title(f"LIME — {top_row['artname']} · {top_row['traname']}")
plt.tight_layout()
plt.show()

Running LIME explanations …
Done.


### 5.3 SHAP vs LIME Agreement Analysis

In [20]:
def _top_pos(val_dict, n=3):
    """Top-n features by absolute magnitude (captures both positive and negative drivers)."""
    return [f for f, v in sorted(val_dict.items(), key=lambda x: abs(x[1]), reverse=True)][:n]

def combined_explanation(track_row, sv_dict, lv_dict, tod, top_n=3):
    """Generate a human-readable explanation combining SHAP and LIME agreement."""
    audio_sv  = {f: sv_dict.get(f, 0) for f in AUDIO_FEATURES}
    audio_lv  = {f: lv_dict.get(f, 0) for f in AUDIO_FEATURES}
    shap_top  = _top_pos(audio_sv, top_n)
    lime_top  = _top_pos(audio_lv, top_n)
    confident = [f for f in shap_top if f in lime_top]
    uncertain = [f for f in shap_top if f not in lime_top]
    lines = []
    if confident:
        labels = [f"{FEATURE_LABELS.get(f,f)} ({'↑' if audio_sv[f]>0 else '↓'})" for f in confident]
        lines.append(f"Primary drivers: {', '.join(labels)}")
    if uncertain:
        labels = [FEATURE_LABELS.get(f, f) for f in uncertain]
        lines.append(f"Supporting (SHAP only): {', '.join(labels)}")
    if not confident and not uncertain:
        lines.append("Broad catalogue match")
    if track_row.get('dominant_tod') == tod:
        lines.append(f"Fits your {tod} listening pattern")
    if track_row.get('total_plays', 0) > tracks['total_plays'].quantile(0.75):
        lines.append("Popular track overall")
    return '\n    '.join(lines)

# Compute agreement rate (Jaccard overlap of top-3 features)
agree_count = 0
total_count = 0
for shap_row, lime_row in zip(shap_rows, lime_rows):
    shap_top3 = _top_pos({f: shap_row.get(f, 0) for f in AUDIO_FEATURES})
    lime_top3 = _top_pos({f: lime_row.get(f, 0) for f in AUDIO_FEATURES})
    overlap   = set(shap_top3) & set(lime_top3)
    union     = set(shap_top3) | set(lime_top3)
    if union:
        agree_count += len(overlap)
        total_count += len(union)

agreement_rate = agree_count / total_count if total_count > 0 else 0
print(f"SHAP vs LIME Agreement Rate: {agreement_rate:.1%}")
print(f"  (Jaccard overlap of top-3 features across {len(shap_rows)} recommendations)")

SHAP vs LIME Agreement Rate: 27.7%
  (Jaccard overlap of top-3 features across 10 recommendations)


In [21]:
# Combined explanation table
def make_table(rows, headers):
    col_widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            for line in str(cell).split('\n'):
                col_widths[i] = max(col_widths[i], len(line))
    def h_line():
        return '+' + '+'.join('-' * (w + 2) for w in col_widths) + '+'
    def format_row(row):
        cell_lines = [str(cell).split('\n') for cell in row]
        max_lines  = max(len(l) for l in cell_lines)
        padded = [lines + [''] * (max_lines - len(lines)) for lines in cell_lines]
        result = []
        for li in range(max_lines):
            result.append('| ' + ' | '.join(padded[i][li].ljust(col_widths[i]) for i in range(len(col_widths))) + ' |')
        return '\n'.join(result)
    lines = [h_line(), format_row(headers), h_line()]
    for row in rows:
        lines.append(format_row(row))
        lines.append(h_line())
    return '\n'.join(lines)

rows = []
for (_, res_row), shap_row, lime_row in zip(hybrid_results.iterrows(), shap_rows, lime_rows):
    shap_top3 = _top_pos({f: shap_row.get(f, 0) for f in AUDIO_FEATURES})
    lime_top3 = _top_pos({f: lime_row.get(f, 0) for f in AUDIO_FEATURES})
    agreed    = sorted(set(shap_top3) & set(lime_top3))
    disputed  = sorted((set(shap_top3) | set(lime_top3)) - set(agreed))
    final_exp = combined_explanation(res_row, shap_row, lime_row, CURRENT_TIME_OF_DAY)
    rows.append([
        f"{res_row['traname']}\n{res_row['artname']}",
        f"{res_row['hybrid_score']:.4f}",
        '\n'.join(shap_top3),
        '\n'.join(lime_top3),
        '\n'.join(agreed)   if agreed   else '-',
        '\n'.join(disputed) if disputed else '-',
        final_exp,
    ])

headers = ['Track / Artist', 'Score', 'SHAP top 3', 'LIME top 3', 'Agree', 'Dispute', 'Explanation']
print(make_table(rows, headers))

+------------------------+--------+------------------+------------------+------------------+------------------+-----------------------------------------------------------------------------+
| Track / Artist         | Score  | SHAP top 3       | LIME top 3       | Agree            | Dispute          | Explanation                                                                 |
+------------------------+--------+------------------+------------------+------------------+------------------+-----------------------------------------------------------------------------+
| Movin' On              | 0.9929 | instrumentalness | valence          | acousticness     | energy           | Primary drivers: acoustic feel (↑)                                          |
| Blur                   |        | acousticness     | loudness         |                  | instrumentalness |     Supporting (SHAP only): instrumental style, high energy                 |
|                        |        | energy        

---
## Stage 6 — Evaluation Layer

Each user's history is split chronologically (80% train / 20% test). Models are retrained on train data only. Ground truth is the set of (artist, track) pairs each user listened to in the test period.

Metrics: **Precision@10**, **Recall@10**, **NDCG@10**, **SHAP–LIME agreement rate**.

In [22]:
def split_user_history(df, train_ratio=0.8):
    """Chronological per-user train/test split."""
    train_list, test_list = [], []
    for uid, group in df.groupby('userid'):
        group = group.sort_values('datetime')
        n     = len(group)
        split = int(n * train_ratio)
        if split < 5 or (n - split) < 1:
            continue
        train_list.append(group.iloc[:split])
        test_list.append(group.iloc[split:])
    return pd.concat(train_list), pd.concat(test_list)

df_train, df_test = split_user_history(df_lastfm)
print(f"Train: {len(df_train):,} rows  |  Test: {len(df_test):,} rows")

# Ground truth: set of (artname, traname) per user in the test period
ground_truth = (
    df_test.groupby('userid')
    .apply(lambda g: set(zip(g['artname'], g['traname'])))
    .to_dict()
)

Train: 399,989 rows  |  Test: 100,011 rows


In [23]:
# Rebuild all models on train data only
merged_train = df_train.merge(
    df_spotify,
    left_on=['artname_norm', 'traname_norm'],
    right_on=['artists_norm', 'name_norm'],
    how='left'
)

tracks_train = merged_train.groupby(['artname', 'traname']).agg(
    total_plays=('userid', 'count'),
    **{f: (f, 'mean') for f in AUDIO_FEATURES}
).reset_index()

for feat in AUDIO_FEATURES:
    artist_means = tracks_train.groupby('artname')[feat].transform('mean')
    tracks_train[feat] = tracks_train[feat].fillna(artist_means)
    tracks_train[feat] = tracks_train[feat].fillna(tracks_train[feat].median())

tracks_train[AUDIO_FEATURES] = scaler.fit_transform(tracks_train[AUDIO_FEATURES])

tod_counts_train = (
    merged_train.groupby(['artname', 'traname', 'time_of_day'])
    .size().unstack(fill_value=0)
    .reindex(columns=TIME_SLOTS, fill_value=0)
)
tod_counts_train = tod_counts_train.div(tod_counts_train.sum(axis=1).replace(0, 1), axis=0)
tracks_train = tracks_train.merge(tod_counts_train.reset_index(), on=['artname', 'traname'], how='left')
tracks_train[TIME_SLOTS] = tracks_train[TIME_SLOTS].fillna(0.25)
tracks_train['track_id'] = range(len(tracks_train))

audio_matrix_tr = csr_matrix(tracks_train[AUDIO_FEATURES].values)
tod_matrix_tr   = csr_matrix(tracks_train[TIME_SLOTS].values)
item_matrix_tr  = hstack([audio_matrix_tr, tod_matrix_tr])

user_item_train  = df_train.groupby(['userid', 'traname']).size().reset_index(name='play_count')
ui_pivot_train   = user_item_train.pivot(index='userid', columns='traname', values='play_count').fillna(0)
ui_sparse_train  = csr_matrix(ui_pivot_train.values)

svd_train           = TruncatedSVD(n_components=50, random_state=42)
user_factors_train  = svd_train.fit_transform(ui_sparse_train)
item_factors_train  = svd_train.components_.T
predicted_train     = user_factors_train @ item_factors_train.T

user_index_train  = {u: i for i, u in enumerate(ui_pivot_train.index)}
track_index_train = {t: i for i, t in enumerate(ui_pivot_train.columns)}

print("Train models rebuilt.")

Train models rebuilt.


In [24]:
def precision_at_k(recommended, truth, k=10):
    rec_set = set(recommended[:k])
    return len(rec_set & truth) / len(rec_set) if rec_set else 0.0

def recall_at_k(recommended, truth, k=10):
    rec_set = set(recommended[:k])
    return len(rec_set & truth) / len(truth) if truth else 0.0

def ndcg_at_k(recommended, truth, k=10):
    dcg = sum(
        1.0 / np.log2(i + 2)
        for i, item in enumerate(recommended[:k])
        if item in truth
    )
    n_relevant = min(len(truth), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(n_relevant))
    return dcg / idcg if idcg > 0 else 0.0

In [25]:
K = 10
eval_users  = [u for u in ground_truth if len(ground_truth[u]) >= 3]
results_eval = {'Content-Based': [], 'Collaborative': [], 'Hybrid': []}

print(f"Evaluating on {len(eval_users)} users with K={K}\n")

def recommend_cb_eval(userid, current_time_of_day, df, tracks, item_matrix, n=10):
    """CB eval variant — scores all tracks (ground truth includes re-listens)."""
    profile, _ = build_user_profile(userid, df, tracks, item_matrix)
    if profile is None:
        return pd.DataFrame()
    candidates = score_candidates(profile, tracks, item_matrix)
    candidates = apply_context_boost(candidates, current_time_of_day)
    return candidates.sort_values('final_score', ascending=False).head(n)

for uid in eval_users:
    truth = ground_truth[uid]

    # Content-Based
    cb     = recommend_cb_eval(uid, 'night', df_train, tracks_train, item_matrix_tr, n=K)
    cb_recs = list(zip(cb['artname'], cb['traname'])) if not cb.empty else []

    # Collaborative Filtering
    cf_recs = []
    if uid in user_index_train:
        u_idx        = user_index_train[uid]
        cf_scores    = predicted_train[u_idx]
        cf_candidates = sorted(
            ((t, cf_scores[t_idx]) for t, t_idx in track_index_train.items()),
            key=lambda x: x[1], reverse=True
        )
        for tname, _ in cf_candidates[:K]:
            match = tracks_train[tracks_train['traname'] == tname]
            if not match.empty:
                cf_recs.append((match.iloc[0]['artname'], tname))

    # Hybrid
    hy = recommend_hybrid(
        uid, 'night', df_train, tracks_train, item_matrix_tr, n=K,
        _user_index=user_index_train,
        _predicted=predicted_train,
        _track_index=track_index_train,
        exclude_heard=False,
    )
    hy_recs = list(zip(hy['artname'], hy['traname'])) if not hy.empty else []

    for name, recs in [('Content-Based', cb_recs), ('Collaborative', cf_recs), ('Hybrid', hy_recs)]:
        results_eval[name].append({
            'userid':    uid,
            'precision': precision_at_k(recs, truth, K),
            'recall':    recall_at_k(recs, truth, K),
            'ndcg':      ndcg_at_k(recs, truth, K),
        })

print(f"{'Model':<20} {'Precision@10':>13} {'Recall@10':>11} {'NDCG@10':>9}")
print('-' * 56)
for model_name, records in results_eval.items():
    df_res = pd.DataFrame(records)
    print(f"{model_name:<20} {df_res['precision'].mean():>13.4f} "
          f"{df_res['recall'].mean():>11.4f} {df_res['ndcg'].mean():>9.4f}")

# SHAP-LIME agreement rate
agree_count = 0
total_count = 0
for shap_row, lime_row in zip(shap_rows, lime_rows):
    shap_top3 = _top_pos({f: shap_row.get(f, 0) for f in AUDIO_FEATURES})
    lime_top3 = _top_pos({f: lime_row.get(f, 0) for f in AUDIO_FEATURES})
    overlap   = set(shap_top3) & set(lime_top3)
    union     = set(shap_top3) | set(lime_top3)
    if union:
        agree_count += len(overlap)
        total_count += len(union)

agreement_rate = agree_count / total_count if total_count > 0 else 0
print(f"\nSHAP vs LIME Agreement Rate: {agreement_rate:.1%}")
print(f"  (Jaccard overlap of top-3 positive features across {len(shap_rows)} recommendations)")

Evaluating on 22 users with K=10

Model                 Precision@10   Recall@10   NDCG@10
--------------------------------------------------------
Content-Based               0.1636      0.0021    0.1661
Collaborative               0.5227      0.0067    0.5197
Hybrid                      0.5000      0.0053    0.5160

SHAP vs LIME Agreement Rate: 27.7%
  (Jaccard overlap of top-3 positive features across 10 recommendations)
